In [1]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Step 1: Dataset load karna
# Seaborn me yeh dataset pehle se hota hai, toh seedhe utha liya.
df = sns.load_dataset('tips')

print("--- Data ki Pehli Jhalak ---")
print(df.head())

# Step 2: Categorical columns ko handle karna (The Key Challenge)
# 'sex', 'smoker', 'day', aur 'time' text columns hain. 
# Model direct text nahi samajhta, isliye pd.get_dummies() use karke inhen 0 aur 1 me badlenge.
# drop_first=True lagane se extra columns create nahi hote (e.g., Male hai toh Female ka alag column nahi banega).
df_encoded = pd.get_dummies(df, columns=['sex', 'smoker', 'day', 'time'], drop_first=True)

print("\n--- Encoding ke baad Data Columns ---")
print(df_encoded.columns)

# Step 3: Features (X) aur Target (y) alag karna
# Target hamara 'tip' hai kyunki hume yahi predict karna hai.
# Baaki bache saare columns hamare features (X) banenge.
X = df_encoded.drop('tip', axis=1)
y = df_encoded['tip']

# Step 4: Data ko Train aur Test sets me baantna
# 80% data training ke liye aur 20% testing ke liye rakh rahe hain.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Model ko Train karna
# Hum RandomForestRegressor use kar rahe hain jo ki ek badhiya ensemble model hai.
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Step 6: Predictions lena
y_pred = model.predict(X_test)

# Step 7: Model ki Performance check karna
# Mean Absolute Error (MAE) batayega ki average me hamari prediction actual tip se kitni door hai.
# R2 Score batayega ki hamara model data ke variance ko kitne acche se samajh pa raha hai.
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- Model Evaluation Results ---")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"R2 Score (Accuracy Metric): {r2:.2f}")

# Step 8: Ek naye customer ke liye khud se predict karke dekhna
# Maan lo ek naya customer aaya: Total Bill = $30, Table Size = 3.
# Aur baaki dummy variables ko humne check kiya (e.g., Male, Non-Smoker, Sunday, Dinner).
sample_data = pd.DataFrame([{
    'total_bill': 30.0,
    'size': 3,
    'sex_Female': 0,      # 0 matlab Male
    'smoker_No': 1,       # 1 matlab Non-smoker
    'day_Fri': 0,
    'day_Sat': 0,
    'day_Sun': 1,         # 1 matlab Sunday
    'time_Dinner': 1      # 1 matlab Dinner time
}])

# Note: Yeh ensure karne ke liye ki columns ka order bilkul wahi ho jo training me tha
sample_data = sample_data.reindex(columns=X.columns, fill_value=0)

predicted_tip = model.predict(sample_data)
print(f"\n[Prediction] $30 ke bill par expected tip hogi lagbhag: ${predicted_tip[0]:.2f}")

--- Data ki Pehli Jhalak ---
   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3
3       23.68  3.31    Male     No  Sun  Dinner     2
4       24.59  3.61  Female     No  Sun  Dinner     4

--- Encoding ke baad Data Columns ---
Index(['total_bill', 'tip', 'size', 'sex_Female', 'smoker_No', 'day_Fri',
       'day_Sat', 'day_Sun', 'time_Dinner'],
      dtype='str')

--- Model Evaluation Results ---
Mean Absolute Error (MAE): $0.77
R2 Score (Accuracy Metric): 0.25

[Prediction] $30 ke bill par expected tip hogi lagbhag: $4.30


# Using Stacking Regression

In [2]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Data Load aur Encoding (Pehle jaisa hi)
df = sns.load_dataset('tips')
df_encoded = pd.get_dummies(df, columns=['sex', 'smoker', 'day', 'time'], drop_first=True)

X = df_encoded.drop('tip', axis=1)
y = df_encoded['tip']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. STUDENT KA JAWAB: Yahan hum define kar rahe hain ki kaun-kaun se models use ho rahe hain.
# Humne ek list banayi jisme har model ko ek 'Naam' (Key) diya hai.
base_models = [
    ('rf_model', RandomForestRegressor(n_estimators=50, random_state=42)),
    ('gb_model', GradientBoostingRegressor(n_estimators=50, random_state=42))
]

# Yeh hamara Meta-Model hai jo dono base models ke predictions ko jodega
meta_model = Ridge()

# 3. Stacking Regressor banana
stacking_model = StackingRegressor(
    estimators=base_models,  # Isme hamare upar wale dono models gaye
    final_estimator=meta_model
)

# Model ko train karna
stacking_model.fit(X_train, y_train)

# 4. Kaise check karein kaunsa model use hua hai? (Code se pata lagana)
print("--- Stacking me use huye Base Models ki List ---")
for name, model in stacking_model.named_estimators_.items():
    print(f"Model Ka Naam: {name} -> Type: {type(model).__name__}")

print(f"\nFinal Meta-Model (Boss): {type(stacking_model.final_estimator_).__name__}")

# 5. Performance Check
y_pred = stacking_model.predict(X_test)
print("\n--- Stacking Model Results ---")
print(f"New R2 Score: {r2_score(y_test, y_pred):.2f}")
print(f"New MAE: ${mean_absolute_error(y_test, y_pred):.2f}")

--- Stacking me use huye Base Models ki List ---
Model Ka Naam: rf_model -> Type: RandomForestRegressor
Model Ka Naam: gb_model -> Type: GradientBoostingRegressor

Final Meta-Model (Boss): Ridge

--- Stacking Model Results ---
New R2 Score: 0.39
New MAE: $0.72
